# Notebook 01 — Ingestion & EDA des données brutes

**Projet :** Juste des Ventilateurs — M2 Data/IA LaPlateforme_  
**Phase :** 2 — Ingestion MQTT et stockage Parquet  
**Objectif :** Explorer les données brutes collectées depuis le simulateur `jumeaux-chauds`.

## Pipeline d'ingestion
```
jumeaux-chauds (FastAPI + MQTT)
    └─► mqtt_subscriber.py   → écoute topics cluster/+/machine/+
        └─► normalizer.py    → normalise les snapshots JSON
            └─► dataset_exporter.py → écrit data/raw/episode=XXX/machine=YYY/part-0.parquet
```

## Structure des données
- `data/raw/episode=XXX/` — données brutes par épisode de simulation
- `data/raw/episode=XXX/metadata.json` — métadonnées (scénario, durée, machines)
- `data/raw/episode=XXX/machine=YYY/part-0.parquet` — série temporelle d'une machine

In [ ]:
import json
import os
import sys
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

warnings.filterwarnings('ignore')

# Racine du projet
_root = Path.cwd()
for _candidate in [_root, _root.parent, _root.parent.parent]:
    if (_candidate / 'data').exists() and (_candidate / 'ingest').exists():
        _root = _candidate
        break
os.chdir(_root)
sys.path.insert(0, str(_root))

RAW_DIR = Path('data/raw')
print(f'Repertoire : {Path.cwd()}')
print(f'Episodes raw disponibles : {sorted(p.name for p in RAW_DIR.glob("episode=*") if p.is_dir())}')

## 1. Métadonnées des épisodes

In [ ]:
episodes = sorted(RAW_DIR.glob('episode=*'), key=lambda p: p.name)
meta_rows = []
for ep in episodes:
    meta_file = ep / 'metadata.json'
    if not meta_file.exists():
        continue
    meta = json.loads(meta_file.read_text())
    machines = [d.name.replace('machine=','') for d in ep.iterdir()
                if d.is_dir() and d.name.startswith('machine=') and d.name != 'machine=_cluster']
    n_records = sum(
        len(pd.read_parquet(f)) for m in machines
        for f in (ep / f'machine={m}').glob('*.parquet')
    )
    meta_rows.append({
        'episode':    meta.get('episode_id', ep.name),
        'scenario':   meta.get('scenario', '?'),
        'duration_s': round(meta.get('duration_s', 0), 1),
        'machines':   len(machines),
        'n_records':  n_records,
        'schema_v':   meta.get('schema_version', '?'),
    })

df_meta = pd.DataFrame(meta_rows)
display(df_meta)

## 2. Schéma des données brutes

In [ ]:
# Charger un exemple de machine
ep_ex  = RAW_DIR / 'episode=001'
df_raw = pd.read_parquet(ep_ex / 'machine=srv-master-01' / 'part-0.parquet')

print(f'Shape : {df_raw.shape}')
print(f'Période : {df_raw["timestamp"].min()} → {df_raw["timestamp"].max()}')
print(f'Fréquence moyenne : {df_raw["timestamp"].diff().dt.total_seconds().median():.1f}s')
print()
display(df_raw.dtypes.to_frame('dtype').assign(
    nulls=df_raw.isnull().sum(),
    null_pct=(df_raw.isnull().mean()*100).round(1)
))

## 3. Séries temporelles clés — épisode 001 (basic)

In [ ]:
# Charger toutes les machines de l'épisode 001
machines = ['srv-master-01','srv-master-02','srv-worker-01','srv-worker-02','srv-worker-03']
dfs = {}
for m in machines:
    f = ep_ex / f'machine={m}' / 'part-0.parquet'
    if f.exists():
        dfs[m] = pd.read_parquet(f).sort_values('timestamp')

fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=True)
colors = ['#e74c3c','#3498db','#2ecc71','#f39c12','#9b59b6']

# Température
ax = axes[0]
for (name, df), c in zip(dfs.items(), colors):
    t = (df['timestamp'] - df['timestamp'].iloc[0]).dt.total_seconds()
    ax.plot(t, df['temperature_c'], label=name, color=c, linewidth=1)
ax.set_ylabel('Temperature (°C)')
ax.set_title('Episode 001 (basic) — Séries temporelles brutes')
ax.legend(fontsize=8)
ax.grid(alpha=0.3)

# RPM fans
ax = axes[1]
for (name, df), c in zip(dfs.items(), colors):
    t = (df['timestamp'] - df['timestamp'].iloc[0]).dt.total_seconds()
    ax.plot(t, df['fan_rpm_mean'], label=name, color=c, linewidth=1)
ax.set_ylabel('Fan RPM moyen')
ax.grid(alpha=0.3)

# Puissance
ax = axes[2]
for (name, df), c in zip(dfs.items(), colors):
    t = (df['timestamp'] - df['timestamp'].iloc[0]).dt.total_seconds()
    ax.plot(t, df['power_w'], label=name, color=c, linewidth=1)
ax.set_ylabel('Puissance (W)')
ax.set_xlabel('Temps (s)')
ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('evaluation/results/fig_01_raw_timeseries.png', dpi=120, bbox_inches='tight')
plt.show()
print('Figure sauvegardee.')

## 4. Comparaison des scénarios

In [ ]:
# Temperature max par épisode (toutes machines)
ep_stats = []
for row in meta_rows:
    ep_id   = row['episode']
    ep_path = RAW_DIR / f'episode={ep_id}'
    temps, rpms = [], []
    for mdir in ep_path.glob('machine=srv-*'):
        for f in mdir.glob('*.parquet'):
            df = pd.read_parquet(f)
            temps.extend(df['temperature_c'].dropna().tolist())
            rpms.extend(df['fan_rpm_mean'].dropna().tolist())
    ep_stats.append({
        'episode':  ep_id,
        'scenario': row['scenario'],
        'T_mean':   round(np.mean(temps), 1) if temps else 0,
        'T_max':    round(np.max(temps), 1)  if temps else 0,
        'T_std':    round(np.std(temps), 2)  if temps else 0,
        'RPM_mean': round(np.mean(rpms), 0)  if rpms else 0,
    })

df_stats = pd.DataFrame(ep_stats).set_index('episode')
display(df_stats)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
scenarios = df_stats['scenario']
x = np.arange(len(df_stats))

ax = axes[0]
bars = ax.bar(x, df_stats['T_max'], color='#e74c3c', alpha=0.8, edgecolor='white')
ax.errorbar(x, df_stats['T_mean'], yerr=df_stats['T_std'], fmt='none',
            color='black', capsize=4, linewidth=1.5, label='mean ± std')
ax.set_xticks(x)
ax.set_xticklabels([f'{ep}\n{sc}' for ep, sc in zip(df_stats.index, scenarios)], fontsize=8)
ax.set_ylabel('Température (°C)')
ax.set_title('T_max et T_mean ± std par épisode')
ax.legend(fontsize=8)
ax.grid(axis='y', alpha=0.3)

ax = axes[1]
ax.bar(x, df_stats['RPM_mean'], color='#3498db', alpha=0.8, edgecolor='white')
ax.set_xticks(x)
ax.set_xticklabels([f'{ep}\n{sc}' for ep, sc in zip(df_stats.index, scenarios)], fontsize=8)
ax.set_ylabel('RPM moyen')
ax.set_title('RPM fan moyen par épisode')
ax.grid(axis='y', alpha=0.3)

plt.suptitle('Comparaison des scénarios — données brutes', fontsize=12)
plt.tight_layout()
plt.savefig('evaluation/results/fig_01_scenario_comparison.png', dpi=120, bbox_inches='tight')
plt.show()

## 5. Distribution des statuts et pannes

In [ ]:
# Distribution des status sur tous les épisodes
all_status = []
all_faults = []
for ep in episodes:
    for mdir in ep.glob('machine=srv-*'):
        for f in mdir.glob('*.parquet'):
            df = pd.read_parquet(f)
            all_status.extend(df['status'].dropna().tolist())
            all_faults.extend(df['fault_types'].dropna().tolist())

from collections import Counter
status_counts = Counter(all_status)
print('Distribution des statuts (toutes machines, tous épisodes):')
total = sum(status_counts.values())
for st, cnt in sorted(status_counts.items(), key=lambda x: -x[1]):
    print(f'  {st:<15} {cnt:>8,}  ({cnt/total*100:.1f}%)')

print()
fault_counts = Counter(all_faults)
print('Types de pannes rencontrées:')
for ft, cnt in sorted(fault_counts.items(), key=lambda x: -x[1]):
    print(f'  {str(ft):<30} {cnt:>6,}')

## 6. Observations et conclusions

### Ce que contiennent les données brutes
- **21 colonnes** par observation : timestamp, identifiants, métriques thermiques, fans, puissance, statut, pannes
- **Fréquence ~5s** entre observations (période de snapshot du simulateur)
- **6 épisodes** couvrant 4 scénarios distincts : `basic`, `busy_weeks`, `heatwave`, `nominal`, `stress`, `trace_replay`

### Points notables
- L'épisode `stress` (005) est le seul avec des shutdowns thermiques et `hot_30s > 0`
- Les colonnes `machines_total` et `machines_on` sont à 100% NaN (données cluster non disponibles par machine)
- `fault_types` est une liste sérialisée — nécessite un parsing dans le feature engineering

### Passage à la Phase 3
Les données brutes sont ensuite enrichies par `features/pipeline.py` :
- Features temporelles (deltas, rolling stats)
- Features contextuelles (time_in_hot_zone, nb_shutdowns)
- Features énergétiques (PUE, energy_fans)
- Labels de supervision (failure_30s, failure_60s, action_class)

→ **Voir notebook 02** pour l'exploration des features calculées.